# Gradient Accumulation

**难度:** Easy

实现微批次梯度累积。

梯度累积通过在多个小批次上累积梯度后执行一次优化器步骤，模拟大批次训练。

**签名:** `accumulated_step(model, optimizer, loss_fn, micro_batches) -> float`

**参数:**
- `model` — nn.Module
- `optimizer` — torch 优化器
- `loss_fn` — 损失函数
- `micro_batches` — (x, y) 元组列表

**返回:** 总损失（浮点数）

**约束:**
- 每个微批次损失在反向传播前除以 `n`
- 必须与单次全批次更新数值一致

**提示:**
1. optimizer.zero_grad() 一次
2. 对每个微批次：前向 → loss/n → 反向
3. optimizer.step()
   除以 n 确保累积梯度与单次全批次更新一致。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [ ]:
# ✅ SOLUTION

def accumulated_step(model, optimizer, loss_fn, micro_batches):
    # ---- Step 1: 清零梯度 ----
    # 在开始累积前清零，只调一次！
    optimizer.zero_grad()

    total_loss = 0.0
    n = len(micro_batches)  # 微批次总数

    # ---- Step 2: 逐个微批次前向+反向 ----
    for x, y in micro_batches:
        # 前向传播
        pred = model(x)
        # 计算损失并除以 n
        # 为什么要除以 n？
        # 因为 loss.backward() 计算的是 d(loss)/d(param)
        # 如果不除以 n，累积的梯度会是单次大 batch 的 n 倍
        # 除以 n 后，累积梯度 = (1/n) * Σ d(loss_i)/d(param) = d(mean_loss)/d(param)
        loss = loss_fn(pred, y) / n
        # 反向传播：梯度累加到 param.grad 上（PyTorch 默认行为）
        loss.backward()
        total_loss += loss.item()

    # ---- Step 3: 统一更新参数 ----
    # 所有微批次的梯度已经累积好了，执行一次参数更新
    optimizer.step()

    return total_loss

In [ ]:
# Demo
model = nn.Linear(4, 2)
opt = torch.optim.SGD(model.parameters(), lr=0.01)
loss = accumulated_step(model, opt, nn.MSELoss(),
    [(torch.randn(2, 4), torch.randn(2, 2)) for _ in range(4)])
print('Accumulated loss:', loss)

In [ ]:
from torch_judge import check
check('gradient_accumulation')

## 📝 核心思路总结

1. **梯度累积 = 小显存模拟大 batch**：在 N 个 micro-batch 上累加梯度，等效于 N 倍大的 batch
2. **loss/n 是关键技巧**：除以 n 保证累积后的梯度与一次性大 batch 计算的梯度数学等价
3. **zero_grad 只调一次**：在累积开始前清零，step 在所有 micro-batch 完成后调用